# Fact Tables Generation

## Purpose:
Create fact tables with all measures and foreign keys to dimensions.

## Fact Tables Created:
1. **fact_orders** - Main fact table (grain: one order)
2. **fact_invoices** - Invoice-level facts (grain: one invoice)
3. **fact_surveys** - Survey response facts (grain: one survey)

## Pre-calculated Metrics:
- Days in shop and cycle times
- Revenue and estimate variance
- On-time delivery flags
- Survey satisfaction scores

In [0]:
from pyspark.sql.functions import (
    col, datediff, coalesce, sum as _sum, count, 
    year, month, to_date, when, lit, round as _round, current_date, abs
)

# Read silver tables
orders = spark.table("mini_project_catalog.02_silver.order")
estimates = spark.table("mini_project_catalog.02_silver.estimate")
stores = spark.table("mini_project_catalog.02_silver.store")

# Read invoices and aggregate by order
invoices = spark.table("mini_project_catalog.02_silver.invoice")
invoice_agg = invoices.groupBy("order_id").agg(
    _sum("invoice_amount").alias("total_invoice_amount"),
    count("invoice_id").alias("invoice_count")
)

# Start building fact table
fact = orders

# Join store details
fact = fact.join(stores, "store_id", "left")

# Join estimates
fact = fact.join(
    estimates.select(
        col("order_id"),
        col("estimate_id"),
        col("estimate_amount"),
        col("estimate_type"),
        col("estimator_id"),
        col("estimator_name")
    ),
    "order_id",
    "left"
)

# Join invoice aggregates
fact = fact.join(invoice_agg, "order_id", "left")

# Calculate derived metrics - Days in shop
fact = fact.withColumn(
    "days_in_shop",
    datediff(coalesce(col("vehicle_out_datetime"), current_date()), col("vehicle_in_datetime"))
)

# Stage-wise cycle times
fact = fact.withColumn(
    "days_vehicle_in_to_work_start",
    datediff(col("actual_work_start_datetime"), col("vehicle_in_datetime"))
)

fact = fact.withColumn(
    "days_work_start_to_completion",
    datediff(col("actual_completion_datetime"), col("actual_work_start_datetime"))
)

fact = fact.withColumn(
    "days_completion_to_delivery",
    datediff(col("actual_delivery_datetime"), col("actual_completion_datetime"))
)

# Delivery metrics
fact = fact.withColumn(
    "delivery_delay_days",
    datediff(col("actual_delivery_datetime"), col("promised_delivery_datetime"))
)

fact = fact.withColumn(
    "delivered_on_time",
    when(col("delivery_delay_days") <= 0, lit(1)).otherwise(lit(0))
)

fact = fact.withColumn(
    "completed_on_time",
    when(
        datediff(col("actual_completion_datetime"), col("planned_completion_datetime")) <= 0,
        lit(1)
    ).otherwise(lit(0))
)

# Extract date dimensions
fact = fact.withColumn("order_year", year(col("vehicle_in_datetime")))
fact = fact.withColumn("order_month", month(col("vehicle_in_datetime")))
fact = fact.withColumn("order_date", to_date(col("vehicle_in_datetime")))

# Estimate accuracy
fact = fact.withColumn(
    "estimate_variance",
    col("total_invoice_amount") - col("estimate_amount")
)

fact = fact.withColumn(
    "estimate_accuracy_pct",
    when(
        col("estimate_amount").isNotNull() & (col("estimate_amount") > 0),
        _round(100 - (abs(col("estimate_variance")) / col("estimate_amount") * 100), 2)
    ).otherwise(lit(None))
)

# Write to gold layer with partitioning
fact.write.mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("order_year", "order_month") \
    .saveAsTable("mini_project_catalog.03_gold.fact_orders")

print(f"✅ fact_orders created with {fact.count()} records")

In [0]:
# Read invoices and join with orders for context
invoices = spark.table("mini_project_catalog.02_silver.invoice")
orders = spark.table("mini_project_catalog.02_silver.order")

fact_invoices = invoices.join(
    orders.select("order_id", "store_id", "service_type", "vehicle_in_datetime"),
    "order_id",
    "left"
)

# Add date dimensions
fact_invoices = fact_invoices.withColumn("invoice_year", year(col("invoice_date")))
fact_invoices = fact_invoices.withColumn("invoice_month", month(col("invoice_date")))
fact_invoices = fact_invoices.withColumn("invoice_date_only", to_date(col("invoice_date")))

# Write to gold layer
fact_invoices.write.mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("invoice_year", "invoice_month") \
    .saveAsTable("mini_project_catalog.03_gold.fact_invoices")

print(f"✅ fact_invoices created with {fact_invoices.count()} records")

In [0]:
# Read surveys and join with orders
surveys = spark.table("mini_project_catalog.02_silver.customer_survey")
orders = spark.table("mini_project_catalog.02_silver.order")

fact_surveys = surveys.join(
    orders.select("order_id", "store_id", "service_type", "technician_id"),
    "order_id",
    "left"
)

# Calculate average survey score
fact_surveys = fact_surveys.withColumn(
    "avg_survey_score",
    when(
        col("responded_flag") == True,
        _round((
            coalesce(col("delivered_on_time_rating"), lit(0)) +
            coalesce(col("work_quality_rating"), lit(0)) +
            coalesce(col("cleanliness_rating"), lit(0)) +
            coalesce(col("communication_rating"), lit(0)) +
            coalesce(col("overall_satisfaction_rating"), lit(0))
        ) / 5, 2)
    ).otherwise(lit(None))
)

# Add date dimensions
fact_surveys = fact_surveys.withColumn("survey_year", year(col("survey_sent_date")))
fact_surveys = fact_surveys.withColumn("survey_month", month(col("survey_sent_date")))

# Write to gold layer
fact_surveys.write.mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("survey_year", "survey_month") \
    .saveAsTable("mini_project_catalog.03_gold.fact_surveys")

print(f"✅ fact_surveys created with {fact_surveys.count()} records")

In [0]:
%sql
SELECT 'fact_orders' as table_name, COUNT(*) as record_count FROM mini_project_catalog.03_gold.fact_orders
UNION ALL
SELECT 'fact_invoices', COUNT(*) FROM mini_project_catalog.03_gold.fact_invoices
UNION ALL
SELECT 'fact_surveys', COUNT(*) FROM mini_project_catalog.03_gold.fact_surveys;